# 04: Solver Validation — Convergence, Energy Conservation, and Ground Truth Certification

**Summary**: We present the complete validation package for our Split-Step Fourier Method
(SSFM) solver. The solver is validated against: (1) analytical solutions, (2) convergence
theory, and (3) conservation laws. The successful completion of all tests establishes the
solver as **trusted ground truth** for the Physics-Informed Neural Network (PINN) project.

**Validation results**:
- 2nd-order convergence verified (log-log slope = 2.0 +/- 0.3)
- Energy conserved near roundoff; validation pass threshold is 10⁻¹⁰
- Gaussian broadening matches analytical formula to < 2%
- Fundamental soliton propagates unchanged (max error < 10⁻⁴)

**Conclusion**: The solver passes all documented validation tests.

## Why Validation Matters

A numerical solver is only as useful as the trust we can place in its output. For our
purposes, this is critical: the solver will generate **training data for a physics-informed
neural network** (PINN-NLSE project). If the training data contains systematic errors,
the PINN will learn wrong physics.

We validate the solver at three levels:

| Level | What We Test | Method |
|-------|-------------|--------|
| **Correctness** | Does the solver reproduce known solutions? | Analytical soliton comparison |
| **Convergence** | Does refining the step size improve accuracy? | Log-log convergence plot |
| **Conservation** | Does the solver preserve the physics (energy)? | E(xi)/E(0) at every step |

In [1]:
import sys
from pathlib import Path
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Run from project root or notebooks/")

sys.path.insert(0, str(ROOT / "src"))

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

from nlse_ssfm.validation import (
    run_convergence_study,
    run_energy_conservation_checks,
    run_gaussian_broadening_check,
    run_soliton_acid_test,
    run_spm_invariance_check,
    run_dispersion_spectral_power_check,
    run_higher_order_soliton_recurrence_checks,
)

## Test 1: Step-Size Convergence

We run the fundamental soliton ($N=1$) at six different step sizes and measure the maximum
amplitude error. The error should scale as $O(d\xi^2)$ — second-order convergence from
Strang (symmetric) splitting.

In [2]:
conv = run_convergence_study(save_path=FIG_DIR / "nb04_convergence.png")
slope = conv['slope']
complex_slope = conv['complex_slope']

print(f"Fitted convergence slope (shape): {slope:.2f}")
print(f"Fitted convergence slope (complex): {complex_slope:.2f}")
print(f"\nConvergence table (N_z, dxi, shape_error, complex_error):")
for row in conv['table']:
    print(f"  N_z={row[0]:5d}, dxi={row[1]:.6f}, "
          f"shape_err={row[2]:.3e}, complex_err={row[3]:.3e}")
assert abs(slope - 2.0) < 0.3
assert abs(complex_slope - 2.0) < 0.4

Fitted convergence slope (shape): 2.00
Fitted convergence slope (complex): 2.00

Convergence table (N_z, dxi, shape_error, complex_error):
  N_z=   50, dxi=0.157080, shape_err=1.213e-03, complex_err=1.256e-03
  N_z=  100, dxi=0.078540, shape_err=3.046e-04, complex_err=3.166e-04
  N_z=  200, dxi=0.039270, shape_err=7.636e-05, complex_err=7.936e-05
  N_z=  500, dxi=0.015708, shape_err=1.223e-05, complex_err=1.271e-05
  N_z= 1000, dxi=0.007854, shape_err=3.057e-06, complex_err=3.177e-06
  N_z= 2000, dxi=0.003927, shape_err=7.644e-07, complex_err=7.945e-07


The fitted shape-error slope confirms second-order convergence, consistent with
the Strang (symmetric) splitting. Halving the step size reduces the splitting error by 4x.

## Test 2: Energy Conservation

The NLSE conserves $E = \int |u|^2 d\tau$. The SSFM preserves this to floating-point
precision because each sub-step is **unitary**:
- Dispersion: FFT -> multiply by $e^{i\phi}$ ($|e^{i\phi}| = 1$) -> IFFT (Parseval)
- Nonlinear: multiply by $e^{iN^2|u|^2 d\xi}$ ($|e^{i\theta}| = 1$)

In [3]:
energy = run_energy_conservation_checks(
    save_path=FIG_DIR / "nb04_energy_conservation.png")
max_E_dev_N1 = energy['max_E_dev_N1']
max_E_dev_N2 = energy['max_E_dev_N2']
print(f"N=1 max energy deviation: {max_E_dev_N1:.2e}")
print(f"N=2 max energy deviation: {max_E_dev_N2:.2e}")
assert max_E_dev_N1 < 1e-10
assert max_E_dev_N2 < 1e-10

N=1 max energy deviation: 1.30e-13
N=2 max energy deviation: 3.16e-13
